In [1]:
# data profiling
import pandas as pd
import os


In [2]:

def load_csv(file_name):
    data_path = 'C:/Users/DELL/Projects/ecommerce-sales-pipeline/data/raw'
    raw=os.path.join(data_path,file_name)
    return pd.read_csv(raw,encoding='latin1', sep=";")

Customers= load_csv('Customers.csv')
Location= load_csv('Location.csv')
Products = load_csv('Products.csv')
Orders= load_csv('Orders.csv')


#reusable csv loader to avoid repeating file-loading logic across datasets
#encoding is set to latin1 because the csv is not encoded in UTF-8, hence errors when reading the csv file 
# use sep=';' so that  we can split fields. But csv are comma-delimited by default

In [3]:
def profile_col(df):
    profile = {
        "Data types": df.dtypes,
        "Missing values":df.isna().sum(),
        "Unique values": df.nunique(),
        "duplicate":df.duplicated().sum()
    }
    return pd.DataFrame(profile)

profile_col(Customers)
# profile_col(Location)
# profile_col(Products)
# profile_col(Orders)


,Data types,Missing values,Unique values,duplicate
Customer ID,object,0,793,0
Customer Name,object,0,793,0


In [4]:
#referential integrity check for customer id in orders 

invalid_orders= Orders[~Orders['Customer ID'].isin(Customers['Customer ID'])]

print(len(invalid_orders))

0


In [5]:
#referential integrity check for product id in orders 

invalid_products= Orders[~Orders['Product ID'].isin(Products['Product ID'])]

print(invalid_products)

      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
16        17  CA-2020-105893  11/11/2020  18/11/2020  Standard Class   
20        21  CA-2020-143336  27/08/2020  01/09/2020    Second Class   
32        33  US-2021-150630  17/09/2021  21/09/2021  Standard Class   
37        38  CA-2021-117415  27/12/2021  31/12/2021  Standard Class   
63        64  CA-2021-135545  24/11/2021  30/11/2021  Standard Class   
...      ...             ...         ...         ...             ...   
9928    9929  CA-2022-129630  04/09/2022  04/09/2022        Same Day   
9956    9957  US-2020-143287  11/11/2020  17/11/2020  Standard Class   
9963    9964  CA-2021-143700  26/07/2021  26/07/2021        Same Day   
9985    9986  CA-2021-100251  17/05/2021  23/05/2021  Standard Class   
9992    9993  CA-2023-121258  26/02/2023  03/03/2023  Standard Class   

     Customer ID      Segment  Postal Code       Product ID    Sales  \
16      PK-19075     Consumer        53711  OFF-ST-10004186  66

In [6]:
#referential integrity check for postal code in orders 

invalid_postal= Orders[~Orders['Postal Code'].isin(Location['Postal Code'])]

print(len(invalid_postal))

0


In [7]:
not_unique =  Orders.duplicated(subset=['Order ID','Product ID']).any()
print(not_unique)

True


In [8]:
def date_summary(date_column,operation):
    latest= date_column.max()
    oldest = date_column.min()

    if operation == 'latest':
       return  latest
    elif operation == 'oldest':
        return oldest
    
    return operation

date_summary(Orders['Ship Date'],'oldest')


'01/01/2021'

In [9]:
#(Orders['Ship Date'] < Orders['Order Date']).any()

In [10]:

#Orders[Orders['Ship Date'] < Orders['Order Date']]


In [11]:
def convert_datetype(date,dateFormat="%d/%m/%Y"):
    appropriate_dateformat = pd.to_datetime(date,format=dateFormat)

    return appropriate_dateformat


Orders['Ship Date'] = convert_datetype(Orders['Ship Date'])
Orders['Order Date'] = convert_datetype(Orders['Order Date'])


# reusable date conversion to avoid repeating multiple date conversions across datasets


In [12]:
(Orders['Ship Date'] < Orders['Order Date']).any()

np.False_

In [13]:
Orders[Orders.duplicated(subset=['Order ID','Product ID'], keep=False)].head(10)


#Why keep=False? By default, duplicated() hides the first instance of a repeated pair. Using keep=False ensures both the original row and its
#duplicates are included in the results so you can inspect them together.

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,Postal Code,Product ID,Sales,Quantity,Discount,Profit
350,351,CA-2022-129714,2022-09-01,2022-09-03,First Class,AB-10060,Home Office,10009,OFF-PA-10001970,24.560,2,0.0,11.5432
352,353,CA-2022-129714,2022-09-01,2022-09-03,First Class,AB-10060,Home Office,10009,OFF-PA-10001970,49.120,4,0.0,23.0864
430,431,US-2022-123750,2022-04-15,2022-04-21,Standard Class,RB-19795,Home Office,28052,TEC-AC-10004659,408.744,7,0.2,76.6395
431,432,US-2022-123750,2022-04-15,2022-04-21,Standard Class,RB-19795,Home Office,28052,TEC-AC-10004659,291.960,5,0.2,54.7425
1300,1301,CA-2022-137043,2022-12-23,2022-12-25,Second Class,LC-17140,Consumer,22153,FUR-FU-10003664,572.760,6,0.0,166.1004
1301,1302,CA-2022-137043,2022-12-23,2022-12-25,Second Class,LC-17140,Consumer,22153,FUR-FU-10003664,286.380,3,0.0,83.0502
3183,3184,CA-2023-152912,2023-11-09,2023-11-12,Second Class,BM-11650,Corporate,21044,OFF-ST-10003208,1633.140,9,0.0,473.6106
3184,3185,CA-2023-152912,2023-11-09,2023-11-12,Second Class,BM-11650,Corporate,21044,OFF-ST-10003208,544.380,3,0.0,157.8702
3405,3406,US-2020-150119,2020-04-23,2020-04-27,Standard Class,LB-16795,Home Office,43229,FUR-CH-10002965,281.372,2,0.3,-12.0588
3406,3407,US-2020-150119,2020-04-23,2020-04-27,Standard Class,LB-16795,Home Office,43229,FUR-CH-10002965,281.372,2,0.3,-12.0588


In [14]:
pro_len=Products[Products.duplicated(subset=['Product ID'])].head(25)
print(pro_len)

           Product ID         Category Sub-Category  \
19    FUR-BO-10002213        Furniture    Bookcases   
67    FUR-CH-10001146        Furniture       Chairs   
186   FUR-FU-10001473        Furniture  Furnishings   
396   OFF-AP-10000576  Office Supplies   Appliances   
516   OFF-AR-10001149  Office Supplies          Art   
732   OFF-BI-10002026  Office Supplies      Binders   
843   OFF-BI-10004632  Office Supplies      Binders   
845   OFF-BI-10004654  Office Supplies      Binders   
1058  OFF-PA-10000357  Office Supplies        Paper   
1064  OFF-PA-10000477  Office Supplies        Paper   
1080  OFF-PA-10000659  Office Supplies        Paper   
1103  OFF-PA-10001166  Office Supplies        Paper   
1162  OFF-PA-10001970  Office Supplies        Paper   
1219  OFF-PA-10003022  Office Supplies        Paper   
1353  OFF-ST-10001228  Office Supplies      Storage   
1441  OFF-ST-10004950  Office Supplies      Storage   
1541  TEC-AC-10002049       Technology  Accessories   
1558  TEC-